In [4]:
from google.colab import drive
import os
import pandas as pd

# 1. Conexión al entorno de Google Drive
drive.mount('/content/drive')

# 2. Definición de rutas según tu arquitectura de Drive
path_raw = '/content/drive/MyDrive/MIA_Tesis_Ecuacalcios/01_Datos/Crudos_Anuales/'
path_models = '/content/drive/MyDrive/MIA_Tesis_Ecuacalcios/02_Modelos/'
path_reports = '/content/drive/MyDrive/MIA_Tesis_Ecuacalcios/03_Reportes/'

# Asegurar la creación de directorios de salida
for folder in [path_models, path_reports]:
    os.makedirs(folder, exist_ok=True)

# 3. Verificación y Carga de Datasets (Soporte XLS/XLSX)
if os.path.exists(path_raw):
    print(f"✅ Carpeta detectada: {path_raw}")

    # Identificación de archivos de Excel (.xls o .xlsx)
    archivos_excel = [f for f in os.listdir(path_raw) if f.lower().endswith(('.xls', '.xlsx'))]
    archivos_excel.sort() # Orden cronológico 2018-2024

    if len(archivos_excel) > 0:
        print(f"📂 Archivos encontrados: {archivos_excel}")

        # Carga masiva mediante compresión de listas
        # Nota: pd.read_excel gestiona internamente el motor según la extensión
        lista_df = []
        for f in archivos_excel:
            file_path = os.path.join(path_raw, f)
            print(f"📖 Cargando: {f}...")
            # Si son .xls antiguos, pd.read_excel usará xlrd automáticamente
            lista_df.append(pd.read_excel(file_path))

        df_raw = pd.concat(lista_df, ignore_index=True)

        print(f"\n🚀 Dataset consolidado con éxito.")
        print(f"📊 Registros totales procesados: {len(df_raw)}")
    else:
        # En caso de que sigan apareciendo como CSV en tu Drive por alguna exportación
        archivos_csv = [f for f in os.listdir(path_raw) if f.lower().endswith('.csv')]
        if len(archivos_csv) > 0:
            print(f"📂 Detectados archivos CSV en su lugar: {archivos_csv}")
            df_raw = pd.concat([pd.read_csv(os.path.join(path_raw, f)) for f in archivos_csv], ignore_index=True)
            print(f"🚀 Dataset consolidado con éxito (formato CSV).")
        else:
            print("❌ Error: No se encontraron archivos XLS o CSV en la ruta.")
else:
    print(f"❌ Error Crítico: La ruta no existe. Verifica la ubicación en tu Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Carpeta detectada: /content/drive/MyDrive/MIA_Tesis_Ecuacalcios/01_Datos/Crudos_Anuales/
📂 Archivos encontrados: ['venta 2018.xls', 'venta 2019.xls', 'venta 2020.xls', 'venta 2021.xls', 'venta 2022.xls', 'venta 2023.xls', 'venta 2024.xls']
📖 Cargando: venta 2018.xls...
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
📖 Cargando: venta 2019.xls...
WARNING *** file size (5753867) not 512 + multiple of sector size (512)
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
📖 Cargando: venta 2020.xls...
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
📖 Cargando: venta 2021.xls...
WARNING *** fi

In [7]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

def procesar_serie_temporal(df):
    """
    Transformación profesional de la data bruta a serie de tiempo mensual.
    """
    # 1. Normalización de la columna 'emision' (Columna H)
    print("⏳ Analizando formatos de fecha en columna 'EMISION'...")

    def parse_fecha(val):
        if pd.isna(val): return pd.NaT
        # Si ya es un objeto de fecha (común en pd.read_excel)
        if isinstance(val, (datetime, pd.Timestamp)):
            return pd.to_datetime(val)
        # Si es un número serial de Excel
        try:
            return datetime(1899, 12, 30) + timedelta(days=float(val))
        except:
            return pd.to_datetime(val, errors='coerce')

    df['fecha_dt'] = df['emision'].apply(parse_fecha)

    # 2. Estandarización del Código de Producto (Columna AD)
    # Convertimos a entero para evitar discrepancias entre '56' y 56.0
    df['codart_int'] = pd.to_numeric(df['codart'], errors='coerce')

    # 3. Filtrado por Producto Crítico: Sulfato de Calcio (56)
    target_code = 56
    df_product = df[df['codart_int'] == target_code].copy()

    if df_product.empty:
        print(f"❌ Alerta: No se encontraron registros para el código {target_code}.")
        print(f"Códigos únicos detectados: {df['codart_int'].unique()[:10]}")
        return None

    # 4. Agregación Mensual (Resampling)
    # Agrupamos por mes para capturar la estacionalidad requerida por la tesis
    df_resampled = df_product.groupby(df_product['fecha_dt'].dt.to_period('M')).agg({
        'cantidad': 'sum',
        'precio': 'mean'
    }).reset_index()

    # 5. Construcción de Features (Variables del Modelo)
    df_resampled['timestamp'] = df_resampled['fecha_dt'].dt.to_timestamp()
    df_resampled['mes_idx'] = df_resampled['timestamp'].dt.month
    df_resampled['trimestre_idx'] = df_resampled['timestamp'].dt.quarter

    return df_resampled

# Ejecución del procesamiento
df_mensual = procesar_serie_temporal(df_raw)

if df_mensual is not None:
    print(f"✅ Procesamiento Finalizado.")
    print(f"📊 Producto detectado: SULFATO DE CALCIO (Código 56)")
    print(f"📈 Puntos de datos mensuales: {len(df_mensual)}")
    display(df_mensual.tail())

⏳ Analizando formatos de fecha en columna 'EMISION'...
✅ Procesamiento Finalizado.
📊 Producto detectado: SULFATO DE CALCIO (Código 56)
📈 Puntos de datos mensuales: 84


,fecha_dt,cantidad,precio,timestamp,mes_idx,trimestre_idx
79,2024-08,34200.0,238.923077,2024-08-01,8,3
80,2024-09,40150.0,457.250000,2024-09-01,9,3
81,2024-10,17500.0,234.857143,2024-10-01,10,4
82,2024-11,17650.0,270.000000,2024-11-01,11,4
83,2024-12,21550.0,137.233333,2024-12-01,12,4


In [8]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_percentage_error
import os

# 1. Definición de Segmentos Cronológicos (Hold-out Method)
# Entrenamiento: Datos históricos consolidados (2018-2023)
# Validación: Datos reales del año actual (2024) para prueba de fuego
train_mask = df_mensual['timestamp'].dt.year <= 2023
test_mask = df_mensual['timestamp'].dt.year == 2024

train_set = df_mensual[train_mask]
test_set = df_mensual[test_mask]

# Variables Independientes (X) y Dependiente (y)
features = ['precio', 'mes_idx', 'trimestre_idx']
target = 'cantidad'

X_train, y_train = train_set[features], train_set[target]
X_test, y_test = test_set[features], test_set[target]

# 2. Parametrización del Algoritmo XGBoost
# Se ajustan hiperparámetros para optimizar la precisión en series de tiempo cortas
model_final = xgb.XGBRegressor(
    n_estimators=1000,          # Número de iteraciones de aprendizaje
    max_depth=4,                # Complejidad de los árboles de decisión
    learning_rate=0.01,         # Velocidad de ajuste (evita sobreajuste)
    subsample=0.8,              # Muestreo aleatorio para robustez
    colsample_bytree=0.8,       # Selección de variables por árbol
    random_state=42             # Garantiza que el resultado sea reproducible
)

print(f"🚀 Entrenando modelo predictivo para Sulfato de Calcio...")
model_final.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

# 3. Inferencia y Cálculo de Métricas de Error (MAPE)
predictions = model_final.predict(X_test)
error_mape = mean_absolute_percentage_error(y_test, predictions)

# 4. Persistencia del Modelo en Google Drive
path_models = '/content/drive/MyDrive/MIA_Tesis_Ecuacalcios/02_Modelos/'
model_output_path = os.path.join(path_models, 'modelo_xgboost_p56.json')
model_final.save_model(model_output_path)

print(f"\n✅ MODELO ENTRENADO Y GUARDADO EN: {path_models}")
print(f"--- RESULTADO DE PRECISIÓN (TEST 2024) ---")
print(f"Precisión del Sistema: {100 - (error_mape * 100):.2f}%")
print(f"Desviación Promedio (MAPE): {error_mape:.4%}")

🚀 Entrenando modelo predictivo para Sulfato de Calcio...

✅ MODELO ENTRENADO Y GUARDADO EN: /content/drive/MyDrive/MIA_Tesis_Ecuacalcios/02_Modelos/
--- RESULTADO DE PRECISIÓN (TEST 2024) ---
Precisión del Sistema: -2.38%
Desviación Promedio (MAPE): 102.3815%
